# Chapter 19 — AI 공급망 보안 — 출처·서명·AI-SBOM

> **Chapter 19 · Part III**

**완성본** — 네 관문(화이트리스트·서명·해시·직렬화 포맷)을 하나의 `load_model()` 로 묶어 모델 공급망 변조를 막습니다.

## 실습 목표
1. 다운로드한 가중치의 해시를 기대값과 대조합니다.
2. pickle 역직렬화 코드실행 위험을 재현하고, safetensors 로 그 위험이 사라짐을 확인합니다.
3. AI-SBOM(매니페스트) 으로 모델·데이터·라이선스 출처를 명세하고, 이 매니페스트 자체를 전자서명 대상으로 삼습니다.
4. 파일과 해시를 함께 바꿔치기하면 해시 검증만으로는 통과함을 실제로 보이고, 서명이 그것을 잡는 것을 확인합니다.
5. 네 관문을 하나의 `load_model()` 로 묶어, 공격 3종(변조·사칭·미승인)이 각각 어느 관문에서 걸리는지 표로 정리합니다.

> **주의:** 데모는 `cryptography`·`safetensors`·`numpy` 외 표준 라이브러리로 동작합니다. pickle 페이로드는 임시 표식 파일 하나만 만드는 무해한 데모이며, 실제 악성 동작은 하지 않습니다.

In [1]:
# 해시 검증·SBOM·화이트리스트·pickle 데모는 표준 라이브러리
# 서명 검증은 cryptography, safetensors 대비는 safetensors + numpy 사용
# (Colab: !pip install -q cryptography safetensors numpy)
# [검증 실행] 생략됨: !pip install -q cryptography safetensors numpy

In [2]:
import hashlib, json, os, pickle, tempfile, pathlib

---
## Step 1. 가중치 해시 검증

공급망 변조(poisoning) 를 잡는 1차 관문입니다.

In [3]:
# === SOLUTION START ===
def verify_weights(data: bytes, expected_sha256: str) -> bool:
    actual = hashlib.sha256(data).hexdigest()
    return actual == expected_sha256
# === SOLUTION END ===
good = b'model-weights-v1'
expected = hashlib.sha256(good).hexdigest()
print('정상 모델:', verify_weights(good, expected))
print('변조 모델:', verify_weights(b'model-weights-v1-TAMPERED', expected))

정상 모델: True
변조 모델: False


---
## Step 2. 직렬화 포맷 — pickle 코드실행 위험과 safetensors

해시·서명은 "받은 파일이 승인된 그 파일인가"를 검증합니다. 그런데 실제 HuggingFace 악성
모델의 다수는 가중치 값이 아니라 **직렬화 포맷의 역직렬화 취약점**을 노립니다(본문 들어가며).
PyTorch 의 기본 체크포인트는 pickle 기반이라, `__reduce__` 를 심어 두면 **로드하는 순간 임의
코드가 실행**됩니다. 아래에서 그 동작을 무해한 표식 파일 생성으로 재현하고, safetensors 는
텐서 데이터만 담아 이 공격이 원천적으로 불가능함을 대비해 봅니다.

In [4]:
# === SOLUTION START ===
# 무해한 데모: __reduce__ 가 표식 파일 하나만 만든다. 실제 공격이라면 이 자리에서
# reverse shell·자격증명 탈취 등 임의 코드가 실행된다.
marker = pathlib.Path(tempfile.gettempdir()) / 'pickle_pwned.marker'
if marker.exists():
    marker.unlink()

class MaliciousCheckpoint:
    """겉보기엔 평범한 모델 객체지만, pickle 로 로드되는 순간 __reduce__ 가 실행된다."""
    def __reduce__(self):
        return (os.system, (f'touch {marker}',))

evil_bytes = pickle.dumps(MaliciousCheckpoint())   # 악성 '체크포인트' 파일에 해당
print('로드 전 표식 파일 존재:', marker.exists())
pickle.loads(evil_bytes)                            # torch.load 가 내부에서 하는 일과 동일
print('로드 후 표식 파일 존재:', marker.exists(), '  <- pickle.load 만으로 코드가 실행됨')
# === SOLUTION END ===

로드 전 표식 파일 존재: False
로드 후 표식 파일 존재: True   <- pickle.load 만으로 코드가 실행됨


In [5]:
# === SOLUTION START ===
# safetensors 는 텐서(숫자 배열)만 저장하는 포맷이라 실행할 코드를 담을 방법이 없다.
import numpy as np
from safetensors.numpy import save_file, load_file

st_path = pathlib.Path(tempfile.gettempdir()) / 'weights.safetensors'
save_file({'layer0.weight': np.zeros(8, dtype=np.float32)}, str(st_path))

marker2 = pathlib.Path(tempfile.gettempdir()) / 'safetensors_pwned.marker'
if marker2.exists():
    marker2.unlink()

loaded = load_file(str(st_path))   # __reduce__ 같은 훅이 없어 코드가 실행될 여지가 없다
print('safetensors 로드 결과 키:', list(loaded.keys()))
print('로드로 인한 표식 파일 존재:', marker2.exists(), '  <- 코드 실행 경로가 아예 없음')
print('그래서 방어의 1순위는 safetensors 같은 코드 실행 불가 포맷 강제입니다.')
# === SOLUTION END ===

safetensors 로드 결과 키: ['layer0.weight']
로드로 인한 표식 파일 존재: False   <- 코드 실행 경로가 아예 없음
그래서 방어의 1순위는 safetensors 같은 코드 실행 불가 포맷 강제입니다.


---
## Step 3. AI-SBOM(매니페스트) 생성 — 서명 대상

AI-SBOM 은 단순히 출력하고 끝나는 문서가 아니라, **다음 단계에서 전자서명할 대상**입니다.
모델·데이터·라이선스와 함께 **Step 1 에서 구한 실제 가중치 해시**를 담아, 이 매니페스트에
서명하면 "가중치 파일 + 출처 메타데이터"가 한꺼번에 위·변조로부터 보호됩니다.

In [6]:
def make_ai_sbom(model, version, dataset, license_, sha):
    # sha 는 Step 1 에서 검증한 실제 가중치 바이트의 해시 — 매니페스트가 파일과 묶인다
    return {'component': model, 'version': version, 'training_data': dataset,
            'license': license_, 'sha256': sha, 'format': 'AI-SBOM/1.0'}

# expected = Step 1 의 정상 가중치(good) 해시
sbom = make_ai_sbom('claude-rag-analyst', '4.5', 'NVD+MITRE', 'internal', expected)
print(json.dumps(sbom, ensure_ascii=False, indent=2))
print('\n이 매니페스트의 sha256 이 실제 가중치 해시와 일치:', sbom['sha256'] == hashlib.sha256(good).hexdigest())

{
  "component": "claude-rag-analyst",
  "version": "4.5",
  "training_data": "NVD+MITRE",
  "license": "internal",
  "sha256": "6903dfbdfe04273eedd148c9cfd6053c7960caa17d0166ac75fde62c8ea6292d",
  "format": "AI-SBOM/1.0"
}

이 매니페스트의 sha256 이 실제 가중치 해시와 일치: True


---
## Step 4. 전자서명 검증 — 해시만으로는 못 막는 위조

해시 검증에는 구멍이 있습니다. 공격자가 **가중치 파일과 매니페스트의 해시값을 함께**
바꿔치기하면, `verify_weights()` 는 "파일 == 매니페스트 해시"만 보므로 그대로 통과합니다.
아래에서 이 전제를 실제로 실행해 확인한 뒤, 전자서명이 그것을 잡는 것을 봅니다. 전자서명은
**보안팀만 보유한 개인키**로 서명하므로, 공개키로 검증에 성공한다는 것은 "보안팀이 이 정확한
내용에 서명했다"는 사실을 보증합니다. Ed25519(비대칭키, 표준 GPG 서명과 동일한 성격의 통제)를
사용합니다.

In [7]:
from cryptography.hazmat.primitives.asymmetric import ed25519
from cryptography.exceptions import InvalidSignature

# === SOLUTION START ===
def sign_manifest(manifest: dict, private_key: ed25519.Ed25519PrivateKey) -> bytes:
    payload = json.dumps(manifest, sort_keys=True).encode()
    return private_key.sign(payload)

def verify_manifest_signature(manifest: dict, signature: bytes,
                               public_key: ed25519.Ed25519PublicKey) -> bool:
    payload = json.dumps(manifest, sort_keys=True).encode()
    try:
        public_key.verify(signature, payload)
        return True
    except (InvalidSignature, ValueError):
        return False
# === SOLUTION END ===

# 보안팀 키 쌍 발급 (실무에서는 HSM/KMS 에 개인키 보관, 공개키만 배포)
security_team_key = ed25519.Ed25519PrivateKey.generate()
security_team_pubkey = security_team_key.public_key()

signature = sign_manifest(sbom, security_team_key)
print('정상 서명 검증:', verify_manifest_signature(sbom, signature, security_team_pubkey))

# --- 전제 실증: 파일과 해시를 함께 바꾸면 해시 검증만으로는 통과한다 ---
tampered_weights = b'model-weights-v1-TAMPERED'
forged_manifest = dict(sbom)
forged_manifest['sha256'] = hashlib.sha256(tampered_weights).hexdigest()  # 해시도 같이 갱신
print('\n[해시 검증만] 변조 가중치 vs 위조 매니페스트 해시:',
      verify_weights(tampered_weights, forged_manifest['sha256']),
      '  <- 파일과 해시를 함께 바꿨으므로 해시검증은 통과(구멍)')

# 서명은 이 위조를 잡는다: 보안팀 개인키가 없어 위조 매니페스트에 유효한 서명을 못 만든다
print('[서명 검증] 위조 매니페스트에 보안팀 서명 있는가:',
      verify_manifest_signature(forged_manifest, signature, security_team_pubkey),
      '  <- 서명이 불일치 → 차단')

# 사칭: 공격자가 자기 키로 위조 서명을 붙여도 보안팀 공개키로는 검증 실패
attacker_key = ed25519.Ed25519PrivateKey.generate()
forged_signature = sign_manifest(forged_manifest, attacker_key)
print('[서명 검증] 공격자 키로 위조 서명(사칭):',
      verify_manifest_signature(forged_manifest, forged_signature, security_team_pubkey),
      '  <- 공개키 불일치 → 차단')

정상 서명 검증: True

[해시 검증만] 변조 가중치 vs 위조 매니페스트 해시: True   <- 파일과 해시를 함께 바꿨으므로 해시검증은 통과(구멍)
[서명 검증] 위조 매니페스트에 보안팀 서명 있는가: False   <- 서명이 불일치 → 차단
[서명 검증] 공격자 키로 위조 서명(사칭): False   <- 공개키 불일치 → 차단


---
## Step 5. 허용 모델 화이트리스트

해시·서명이 모두 유효해도 애초에 **승인되지 않은 모델**을 로드하면 안 됩니다. 화이트리스트는
"이 값이 신뢰할 수 있게 서명됐는가" 이전에 "이 모델을 애초에 쓸 수 있는가"를 걸러내는
가장 앞단의 관문입니다.

In [8]:
# === SOLUTION START ===
ALLOWED_MODELS = {
    'claude-rag-analyst': '4.5',
    'llama-3.2-3b-instruct': '3.2',
    'gpt-4o-mini': '2024-07-18',
}

def is_allowed_model(model_name: str) -> bool:
    return model_name in ALLOWED_MODELS
# === SOLUTION END ===

print('허용 모델(claude-rag-analyst):', is_allowed_model('claude-rag-analyst'))
print('비허용 모델(random-huggingface-model):', is_allowed_model('random-huggingface-model'))

허용 모델(claude-rag-analyst): True
비허용 모델(random-huggingface-model): False


---
## Step 6. 네 관문을 하나로 — `load_model()` 통합 게이트

네 관문이 따로 놀면 "승인된 이름 + 변조된 가중치 + 함께 위조한 매니페스트"가 어느 한 검사만
통과해도 로드될 수 있습니다. 그래서 **화이트리스트 → 서명 → 해시**를 하나의 `load_model()`
로 묶어, 하나라도 실패하면 로드를 거부합니다. safetensors 포맷 강제(Step 2)는 이 게이트에
들어오기 전에 파일 확장자/매직으로 거르는 별도 관문으로 둡니다. 아래에서 공격 3종이 각각
어느 관문에서 걸리는지 확인합니다.

In [9]:
# === SOLUTION START ===
class ModelRejected(Exception):
    pass

def load_model(name, weights: bytes, manifest: dict, signature: bytes, pubkey):
    """화이트리스트 → 서명 → 해시 순서로 검증하고, 통과한 것만 '로드'한다.
    어느 관문에서 막혔는지 예외 메시지로 돌려준다."""
    # 관문 1: 화이트리스트 — 애초에 쓸 수 있는 모델인가
    if name not in ALLOWED_MODELS:
        raise ModelRejected('화이트리스트')
    # 관문 2: 서명 — 보안팀이 이 매니페스트에 서명했는가 (매니페스트 위조·사칭 차단)
    if not verify_manifest_signature(manifest, signature, pubkey):
        raise ModelRejected('서명')
    # 관문 3: 해시 — 실제 가중치가 (서명된) 매니페스트의 해시와 일치하는가
    if not verify_weights(weights, manifest['sha256']):
        raise ModelRejected('해시')
    return f"{name} 로드 허용"

def try_load(label, name, weights, manifest, signature, pubkey):
    try:
        print(f"{label:38} → {load_model(name, weights, manifest, signature, pubkey)}")
        return '통과(허용)'
    except ModelRejected as e:
        print(f"{label:38} → 차단: {e} 관문")
        return str(e)
# === SOLUTION END ===

# 정상 케이스: 화이트리스트 등재 + 보안팀 서명 + 해시 일치
r_ok = try_load('정상 모델', 'claude-rag-analyst', good, sbom, signature, security_team_pubkey)

# 공격 1 — 변조: 승인된 이름, 가중치만 변조, 매니페스트는 원본 그대로 (해시 불일치)
r_tamper = try_load('변조(가중치만)', 'claude-rag-analyst', tampered_weights, sbom, signature, security_team_pubkey)

# 공격 2 — 사칭: 가중치·매니페스트를 함께 위조하고 공격자 키로 서명
r_forge = try_load('사칭(공격자 서명)', 'claude-rag-analyst', tampered_weights, forged_manifest, forged_signature, security_team_pubkey)

# 공격 3 — 미승인: 화이트리스트에 없는 모델 (서명이 아무리 유효해도 진입 자체를 차단)
r_unapproved = try_load('미승인 모델', 'evil/backdoored-llm', good, sbom, signature, security_team_pubkey)

정상 모델                                  → claude-rag-analyst 로드 허용
변조(가중치만)                               → 차단: 해시 관문
사칭(공격자 서명)                             → 차단: 서명 관문
미승인 모델                                 → 차단: 화이트리스트 관문


In [10]:
# 공격 3종이 각각 어느 관문에서 걸리는지 정리
rows = [
    ('변조(가중치만)', '승인된 이름 + 변조 가중치, 매니페스트 원본', r_tamper),
    ('사칭(위조+서명)', '가중치·매니페스트 함께 위조, 공격자 키 서명', r_forge),
    ('미승인', '화이트리스트에 없는 모델', r_unapproved),
]
print(f"{'공격':16}{'시나리오':44}{'차단 관문'}")
print('-' * 88)
for atk, scenario, gate in rows:
    print(f"{atk:16}{scenario:44}{gate}")
print('\n정상 모델 결과:', r_ok)

공격              시나리오                                        차단 관문
----------------------------------------------------------------------------------------
변조(가중치만)        승인된 이름 + 변조 가중치, 매니페스트 원본                   해시
사칭(위조+서명)       가중치·매니페스트 함께 위조, 공격자 키 서명                   서명
미승인             화이트리스트에 없는 모델                               화이트리스트

정상 모델 결과: 통과(허용)


---
## 정리
- **허용 모델 화이트리스트 · 서명 검증 · 해시 매니페스트 · 코드실행 불가 포맷(safetensors)** 이
  모델 공급망 변조를 막는 기본 통제입니다. 이 노트북은 이를 각각 구현한 뒤 `load_model()`
  하나로 묶어(Step 6), 관문이 따로 놀 때 생기는 "승인된 이름 + 변조된 가중치" 통과 구멍을 닫았습니다.
- 해시만으로는 파일·해시를 함께 바꾼 위조를 못 막지만(Step 4), 서명은 보안팀 개인키 없이는
  위조가 불가능해 그 구멍을 막습니다.
- pickle 체크포인트는 로드만으로 코드가 실행되므로(Step 2), 방어의 1순위는 safetensors 같은
  코드 실행 불가 포맷 강제입니다.
- N=1 변조가 N=수천 파생 모델로 증폭되는 위험(ch04 공급망)을 차단합니다.

> **다음 챕터:** LLM Sandbox.